# Home Credit Default Risk
## Sprint 4 — Análise Final, Interpretabilidade e Fechamento do Ciclo

**Período:** 15/06 a 21/06  

Esta sprint fecha o ciclo do projeto: não construímos mais modelos, mas entendemos em profundidade o que foi construído, comunicamos com clareza e documentamos com responsabilidade.

**Estrutura do notebook:**

| Seção | Conteúdo |
|---|---|
| 0 | Reprodução compacta do pipeline da Sprint 3 → dados e modelo prontos |
| 1 | Análise dos erros do modelo — perfil dos Falsos Negativos |
| 2 | Interpretação das features mais importantes (Feature Importance + SHAP) |
| 3 | Revisão crítica das hipóteses da Sprint 1 |
| 4 | Tabela consolidada de desempenho (todas as sprints) |
| 5 | Limitações honestas e melhorias possíveis |

---
## SEÇÃO 0 — Reprodução do Pipeline da Sprint 3

Reproduzimos o pipeline completo de pré-processamento (Sprint 2) e carregamos o modelo salvo (Sprint 3). Todo `fit()` é executado apenas no conjunto de treino — sem data leakage.

> **Importante:** o índice de `y_test` aponta para as linhas originais do `df_raw`, o que nos permite recuperar os dados brutos dos casos de erro (FN/FP) para análise demográfica na Seção 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

print('Bibliotecas carregadas.')
print(f'SHAP versão: {shap.__version__}')

In [ ]:
# ─── 0.1 Carregamento e Split ─────────────────────────────────────────────────
df_raw = pd.read_csv('../data/raw/application_train.csv')
X = df_raw.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_raw['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# ─── 0.2 Sentinela + Missing ──────────────────────────────────────────────────
X_train['DAYS_EMPLOYED'] = X_train['DAYS_EMPLOYED'].replace(365243, np.nan)
X_test['DAYS_EMPLOYED']  = X_test['DAYS_EMPLOYED'].replace(365243, np.nan)

miss_pct  = X_train.isnull().mean()
cols_drop = miss_pct[miss_pct > 0.60].index.tolist()
X_train = X_train.drop(columns=cols_drop)
X_test  = X_test.drop(columns=cols_drop)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
med_imp  = SimpleImputer(strategy='median')
X_train[num_cols] = med_imp.fit_transform(X_train[num_cols])
X_test[num_cols]  = med_imp.transform(X_test[num_cols])

cat_cols = X_train.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    moda = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(moda)
    X_test[col]  = X_test[col].fillna(moda)

# ─── 0.3 Outliers — Winsorização IQR ─────────────────────────────────────────
skip_win = {'HOUR_APPR_PROCESS_START'}
for col in X_train.select_dtypes(include=[np.number]).columns:
    if col in skip_win:
        continue
    Q1, Q3 = X_train[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    if IQR == 0:
        continue
    if ((X_train[col] < Q1 - 1.5*IQR) | (X_train[col] > Q3 + 1.5*IQR)).mean() > 0.10:
        continue
    X_train[col] = X_train[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
    X_test[col]  = X_test[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# ─── 0.4 Encoding ────────────────────────────────────────────────────────────
ordinal_maps = {
    'NAME_EDUCATION_TYPE': ['Lower secondary', 'Secondary / secondary special',
                            'Incomplete higher', 'Higher education', 'Academic degree'],
    'HOUSETYPE_MODE':      ['terraced house', 'specific housing', 'block of flats'],
}
for col, order in ordinal_maps.items():
    if col not in X_train.columns:
        continue
    mapping = {cat: idx for idx, cat in enumerate(order)}
    X_train[col] = X_train[col].map(mapping)
    X_test[col]  = X_test[col].map(mapping)

ohe_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
            'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START',
            'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
ohe_cols = [c for c in ohe_cols if c in X_train.columns]
X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=ohe_cols, drop_first=True, dtype=int)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

te_cols = [c for c in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE'] if c in X_train.columns]
for col in te_cols:
    te_map = y_train.groupby(X_train[col]).mean()
    X_train[col] = X_train[col].map(te_map)
    X_test[col]  = X_test[col].map(te_map).fillna(y_train.mean())

# ─── 0.5 Feature Engineering ──────────────────────────────────────────────────
for df_ in [X_train, X_test]:
    df_['CREDIT_TO_INCOME']  = df_['AMT_CREDIT']  / (df_['AMT_INCOME_TOTAL'] + 1)
    df_['ANNUITY_TO_INCOME'] = df_['AMT_ANNUITY'] / (df_['AMT_INCOME_TOTAL'] + 1)
    df_['CREDIT_TO_ANNUITY'] = df_['AMT_CREDIT']  / (df_['AMT_ANNUITY'] + 1)
    df_['EMPLOYMENT_DAYS']   = -df_['DAYS_EMPLOYED'] / 365.25

# ─── 0.6 Scaling ─────────────────────────────────────────────────────────────
num_final = X_train.select_dtypes(include=[np.number]).columns.tolist()
feat_robust, feat_std = [], []
for col in num_final:
    Q1, Q3 = X_train[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    if IQR == 0:
        feat_std.append(col)
        continue
    out_pct = ((X_train[col] < Q1 - 1.5*IQR) | (X_train[col] > Q3 + 1.5*IQR)).mean()
    (feat_robust if out_pct > 0.05 else feat_std).append(col)

if feat_std:
    sc_std = StandardScaler()
    X_train[feat_std] = sc_std.fit_transform(X_train[feat_std])
    X_test[feat_std]  = sc_std.transform(X_test[feat_std])
if feat_robust:
    sc_rob = RobustScaler()
    X_train[feat_robust] = sc_rob.fit_transform(X_train[feat_robust])
    X_test[feat_robust]  = sc_rob.transform(X_test[feat_robust])

# ─── 0.7 Seleção de Features ─────────────────────────────────────────────────
sel_var      = VarianceThreshold(threshold=0.01)
sel_var.fit(X_train)
low_var_cols = X_train.columns[~sel_var.get_support()].tolist()

corr_abs      = X_train.corrwith(y_train).abs()
low_corr_cols = corr_abs[corr_abs < 0.005].index.tolist()

remove_init = list(set(low_var_cols) | set(low_corr_cols))
X_pf      = X_train.drop(columns=remove_init, errors='ignore')
X_pf_test = X_test.drop(columns=remove_init,  errors='ignore')

rf_sel = RandomForestClassifier(
    n_estimators=30, max_depth=5, n_jobs=-1,
    random_state=RANDOM_STATE, class_weight='balanced'
)
rf_sel.fit(X_pf, y_train)

imp_df = pd.DataFrame({'feature': X_pf.columns, 'importance': rf_sel.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

fe_feats = ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_ANNUITY', 'EMPLOYMENT_DAYS']
top60    = imp_df.head(60)['feature'].tolist()
selected = list(dict.fromkeys(top60 + [f for f in fe_feats if f in X_pf.columns and f not in top60]))
selected = [f for f in selected if f in X_pf.columns]

X_train_final = X_pf[selected].copy()
X_test_final  = X_pf_test[selected].copy()

print('Pipeline reproduzido com sucesso.')
print(f'X_train_final: {X_train_final.shape} | X_test_final: {X_test_final.shape}')
print(f'TARGET=1 no treino: {y_train.mean():.2%} | no teste: {y_test.mean():.2%}')

In [ ]:
# ─── 0.8 Carregar modelo + re-treinar no conjunto completo ───────────────────
melhor_pipeline = joblib.load('modelo_projeto.pkl')

# Re-treinar nos 246k para garantir consistência (o pkl pode ter sido salvo com 50k)
melhor_pipeline.fit(X_train_final, y_train)

modelo_nome = melhor_pipeline.named_steps['model'].__class__.__name__

y_pred       = melhor_pipeline.predict(X_test_final)
y_pred_proba = melhor_pipeline.predict_proba(X_test_final)[:, 1]

auc_test  = roc_auc_score(y_test, y_pred_proba)
f1_test   = f1_score(y_test, y_pred, average='macro')
acc_test  = accuracy_score(y_test, y_pred)
prec_1    = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec_1     = recall_score(y_test, y_pred, pos_label=1, zero_division=0)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'Modelo carregado: {modelo_nome}')
print(f'AUC-ROC : {auc_test:.4f}')
print(f'F1-Macro: {f1_test:.4f}')
print(f'Acurácia: {acc_test:.4f}')
print(f'\nMatriz de Confusão:')
print(f'  TN={tn:,}  FP={fp:,}')
print(f'  FN={fn:,}  TP={tp:,}')
print(f'\nFalsos Negativos (inadimplentes não detectados): {fn:,}')
print(f'Recall classe 1 (taxa de detecção): {rec_1:.1%}')

---
## SEÇÃO 1 — Análise dos Erros do Modelo

A análise de erros vai além da matriz de confusão: queremos entender **quem** são os casos classificados incorretamente e **por quê** o modelo falha neles.

**Foco principal: Falsos Negativos (FN)**

Os FN são inadimplentes que o modelo classificou como bons pagadores — o erro mais custoso em crédito, pois representa **perda financeira direta** (capital emprestado não recuperado). O banco aprova o crédito, libera o dinheiro e não recebe de volta.

**Perguntas a responder:**
1. Quantos FN existem e qual sua proporção em relação ao total de inadimplentes?
2. Qual o perfil demográfico dos FN? (gênero, educação, tipo de renda)
3. Qual o perfil financeiro dos FN? (renda, valor de crédito, anuidade)
4. O que diferencia um FN de um TP (inadimplente corretamente detectado)?

**Metodologia:** Usamos o índice do `y_test` para recuperar os registros originais do `df_raw`, preservando as labels categóricas legíveis antes do encoding.

In [ ]:
# Identificar os 4 grupos de erro
idx_test = y_test.index

mask_tn = (y_pred == 0) & (y_test.values == 0)
mask_fp = (y_pred == 1) & (y_test.values == 0)
mask_fn = (y_pred == 0) & (y_test.values == 1)  # FOCO PRINCIPAL
mask_tp = (y_pred == 1) & (y_test.values == 1)

idx_fn = idx_test[mask_fn]
idx_tp = idx_test[mask_tp]
idx_fp = idx_test[mask_fp]

# Recuperar dados brutos dos grupos (antes do encoding)
df_fn = df_raw.loc[idx_fn].copy()
df_tp = df_raw.loc[idx_tp].copy()
df_fp = df_raw.loc[idx_fp].copy()
df_inadimplentes = df_raw.loc[idx_test[y_test.values == 1]].copy()

print('=' * 65)
print('RESUMO DOS ERROS DO MODELO')
print('=' * 65)
total_test      = len(y_test)
total_inad_test = (y_test == 1).sum()

print(f'Total no conjunto de teste          : {total_test:,}')
print(f'Total de inadimplentes reais        : {total_inad_test:,} ({total_inad_test/total_test:.1%})')
print()
print(f'Verdadeiro Negativo (TN)            : {tn:,} ({tn/total_test:.1%}) — adimplentes corretamente aprovados')
print(f'Falso Positivo (FP)                 : {fp:,} ({fp/total_test:.1%}) — adimplentes negados (custo de oportunidade)')
print(f'Falso Negativo (FN) ⚠               : {fn:,} ({fn/total_test:.1%}) — inadimplentes aprovados (PERDA DIRETA)')
print(f'Verdadeiro Positivo (TP)            : {tp:,} ({tp/total_test:.1%}) — inadimplentes detectados corretamente')
print()
print(f'Dos {total_inad_test:,} inadimplentes reais:')
print(f'  Detectados (TP)    : {tp:,} ({tp/total_inad_test:.1%})')
print(f'  Não detectados (FN): {fn:,} ({fn/total_inad_test:.1%}) ← foco desta análise')

In [ ]:
# Visualização da composição de erros
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Matriz de Confusão
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Adimplente (0)', 'Inadimplente (1)']
)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusão — {modelo_nome}', fontsize=11, fontweight='bold')

# Gráfico 2: Composição dos inadimplentes — detectados vs não detectados
cores = ['#2ecc71', '#e74c3c']
valores = [tp, fn]
labels_pie = [f'Detectados (TP)\n{tp:,} ({tp/total_inad_test:.1%})',
              f'Não detectados (FN)\n{fn:,} ({fn/total_inad_test:.1%})']
axes[1].pie(valores, labels=labels_pie, colors=cores, autopct='', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Inadimplentes Reais: Detectados vs Não Detectados', fontsize=11, fontweight='bold')

plt.suptitle('Análise dos Erros — Conjunto de Teste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição das probabilidades preditas: FN vs TP
proba_fn = y_pred_proba[mask_fn]
proba_tp = y_pred_proba[mask_tp]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de probabilidades
axes[0].hist(proba_fn, bins=30, color='#e74c3c', alpha=0.7, label=f'FN (n={fn:,})', density=True)
axes[0].hist(proba_tp, bins=30, color='#2ecc71', alpha=0.7, label=f'TP (n={tp:,})', density=True)
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.50')
axes[0].set_xlabel('P(inadimplência) predita pelo modelo')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Distribuição de Probabilidade: FN vs TP', fontsize=11, fontweight='bold')
axes[0].legend()

# Boxplot
data_box = pd.DataFrame({
    'Probabilidade': np.concatenate([proba_fn, proba_tp]),
    'Grupo': ['FN'] * len(proba_fn) + ['TP'] * len(proba_tp)
})
sns.boxplot(data=data_box, x='Grupo', y='Probabilidade',
            palette={'FN': '#e74c3c', 'TP': '#2ecc71'}, ax=axes[1])
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.50')
axes[1].set_title('Probabilidade Mediana: FN vs TP', fontsize=11, fontweight='bold')
axes[1].set_ylabel('P(inadimplência)')
axes[1].legend()

plt.suptitle('Por que os FN são difíceis de detectar?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Probabilidade média FN : {proba_fn.mean():.4f} (threshold = 0.50 → classificado como 0)')
print(f'Probabilidade média TP : {proba_tp.mean():.4f}')
print(f'\nOs FN são inadimplentes que o modelo atribui baixa probabilidade de risco — casos que')
print(f'se comportam financeiramente de forma parecida com adimplentes no conjunto de features disponíveis.')

In [ ]:
# Perfil demográfico: FN vs TP vs adimplentes
df_adimplentes = df_raw.loc[idx_test[y_test.values == 0]].copy()

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

grupos = {
    'FN (Inad. Não Detectado)': (df_fn, '#e74c3c'),
    'TP (Inad. Detectado)':     (df_tp, '#2ecc71'),
    'Adimplentes':              (df_adimplentes, '#3498db'),
}

# 1. Gênero
ax = axes[0]
for nome, (df_, cor) in grupos.items():
    gen_pct = df_['CODE_GENDER'].value_counts(normalize=True)
    gen_f   = gen_pct.get('F', 0) * 100
    ax.bar(nome, gen_f, color=cor, alpha=0.85, edgecolor='black')
    ax.text(list(grupos.keys()).index(nome), gen_f + 0.5, f'{gen_f:.1f}%', ha='center', fontsize=9)
ax.set_title('% Feminino', fontsize=10, fontweight='bold')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=12)
ax.set_ylim(0, 100)

# 2. Tipo de Renda
ax = axes[1]
renda_compare = {}
for nome, (df_, cor) in grupos.items():
    renda_compare[nome] = df_['NAME_INCOME_TYPE'].value_counts(normalize=True) * 100
renda_df = pd.DataFrame(renda_compare).fillna(0)
renda_df.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', alpha=0.85)
ax.set_title('Distribuição de Tipo de Renda (%)', fontsize=10, fontweight='bold')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=15)
ax.legend(fontsize=7, loc='upper right')

# 3. Nível de Educação
ax = axes[2]
edu_order = ['Lower secondary', 'Secondary / secondary special',
             'Incomplete higher', 'Higher education', 'Academic degree']
edu_compare = {}
for nome, (df_, cor) in grupos.items():
    edu_compare[nome] = df_['NAME_EDUCATION_TYPE'].value_counts(normalize=True) * 100
edu_df = pd.DataFrame(edu_compare).reindex(edu_order).fillna(0)
edu_df.T.plot(kind='bar', ax=ax, colormap='Set1', edgecolor='black', alpha=0.85)
ax.set_title('Nível de Educação (%)', fontsize=10, fontweight='bold')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=15)
ax.legend(fontsize=7)

# 4. Renda total
ax = axes[3]
for nome, (df_, cor) in grupos.items():
    renda = df_['AMT_INCOME_TOTAL'].clip(0, df_['AMT_INCOME_TOTAL'].quantile(0.99))
    ax.hist(renda, bins=40, color=cor, alpha=0.5, label=nome, density=True)
ax.set_title('Distribuição de Renda Total', fontsize=10, fontweight='bold')
ax.set_xlabel('Renda Total (R$)')
ax.legend(fontsize=8)

# 5. Valor do Crédito
ax = axes[4]
for nome, (df_, cor) in grupos.items():
    cred = df_['AMT_CREDIT'].clip(0, df_['AMT_CREDIT'].quantile(0.99))
    ax.hist(cred, bins=40, color=cor, alpha=0.5, label=nome, density=True)
ax.set_title('Distribuição do Valor do Crédito', fontsize=10, fontweight='bold')
ax.set_xlabel('Valor do Crédito (R$)')
ax.legend(fontsize=8)

# 6. Dias de Emprego (estabilidade)
ax = axes[5]
for nome, (df_, cor) in grupos.items():
    emp = df_['DAYS_EMPLOYED'].replace(365243, np.nan).dropna()
    emp_anos = (-emp / 365).clip(0, 30)
    ax.hist(emp_anos, bins=40, color=cor, alpha=0.5, label=nome, density=True)
ax.set_title('Anos de Emprego (estabilidade)', fontsize=10, fontweight='bold')
ax.set_xlabel('Anos empregado')
ax.legend(fontsize=8)

plt.suptitle('Perfil Demográfico e Financeiro: FN vs TP vs Adimplentes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tabela comparativa de métricas financeiras
metricas_financeiras = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_EMPLOYED']

def sumarizar(df_, cols):
    rows = {}
    for col in cols:
        vals = df_[col].replace(365243, np.nan).dropna()
        rows[col] = {
            'Mediana': round(vals.median(), 0),
            'Média':   round(vals.mean(),   0),
            'Std':     round(vals.std(),    0),
        }
    return pd.DataFrame(rows).T

print('Comparação de métricas financeiras: FN vs TP vs Adimplentes')
print('=' * 65)
for nome, (df_, _) in grupos.items():
    print(f'\n--- {nome} (n={len(df_):,}) ---')
    display(sumarizar(df_, metricas_financeiras))

# Proporção por tipo de renda
print('\n' + '=' * 65)
print('Taxa de inadimplência real por tipo de renda (nos FN+TP combinados):')
inad_real = pd.concat([df_fn.assign(grupo='FN'), df_tp.assign(grupo='TP')])
taxa_tipo = inad_real.groupby('NAME_INCOME_TYPE')['grupo'].apply(
    lambda x: (x == 'FN').sum() / len(x) * 100
).sort_values(ascending=False).round(1)
print('% de FN dentro de cada tipo de renda (dos inadimplentes):')
print(taxa_tipo.to_string())

### 1.1 Interpretação dos Resultados — Perfil dos Falsos Negativos

Os Falsos Negativos representam o subconjunto de inadimplentes que o modelo considera "seguros" — eles têm características que os tornam indistinguíveis de bons pagadores nas features disponíveis. Os padrões observados têm implicações diretas para a política de concessão de crédito:

**Por que o modelo falha neles:**
- Os FN apresentam probabilidade predita **abaixo de 0.50**, o que significa que o modelo não tem sinal suficiente nesse grupo para superar o threshold de decisão.
- A distribuição de probabilidade dos FN se concentra na faixa baixa (< 0.5), ao contrário dos TP, que têm probabilidades mais altas — confirmando que esses casos são "limítrofes" do ponto de vista do modelo.

**Implicação para política de crédito:**
- Uma estratégia de redução de FN é **abaixar o threshold** de 0.50 para, por exemplo, 0.35 ou 0.40: qualquer solicitante com P(inadimplência) > threshold seria sinalizado para revisão manual.
- Este trade-off aumenta os FP (bons pagadores negados), mas reduz a exposição financeira direta, o que é o objetivo prioritário em contextos de risco de crédito.
- O ponto ótimo do threshold é uma decisão de negócio que depende do custo relativo de FN vs FP — explorado na Seção 5.

---
## SEÇÃO 2 — Interpretação das Features Mais Importantes

Duas camadas de interpretabilidade são apresentadas:

| Técnica | O que revela | Limitação |
|---|---|---|
| **Feature Importance** (nativa) | Importância global agregada de cada feature no modelo | Não indica direção (positiva/negativa) nem interações |
| **SHAP (SHapley Additive exPlanations)** | Contribuição individual de cada feature para cada predição | Computacionalmente mais caro |

**Por que SHAP é essencial em crédito:**
- Regulamentações como Basel III exigem que as instituições financeiras consigam explicar individualmente por que um crédito foi negado.
- SHAP fornece exatamente isso: "A renda baixa contribuiu -0.12 para a probabilidade de inadimplência deste cliente".
- Features com alta importância global mas direção negativa (reduzem risco) são tão valiosas para a política quanto as que aumentam risco.

In [ ]:
# Feature importance nativa do modelo
from sklearn.ensemble import (HistGradientBoostingClassifier, RandomForestClassifier,
                               GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

clf = melhor_pipeline.named_steps['model']
feature_names = X_test_final.columns.tolist()

_tree_types   = (HistGradientBoostingClassifier, RandomForestClassifier,
                 GradientBoostingClassifier, DecisionTreeClassifier)
_linear_types = (LogisticRegression,)

if isinstance(clf, _tree_types) and hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    imp_label   = 'Feature Importance (MDI — Mean Decrease Impurity)'
    imp_type    = 'tree'
elif isinstance(clf, _linear_types):
    importances = np.abs(clf.coef_[0])
    imp_label   = '|Coeficiente| — Regressão Logística'
    imp_type    = 'linear'
else:
    # HGBT sem feature_importances_ acessível: usar permutation importance proxy
    importances = np.ones(len(feature_names))
    imp_label   = 'Importância uniforme (feature_importances_ indisponível nesta versão do sklearn)'
    imp_type    = 'tree'  # ainda é tree para o SHAP

imp_df_final = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df_final = imp_df_final.sort_values('importance', ascending=False).reset_index(drop=True)

TOP = 25
top_imp = imp_df_final.head(TOP)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_imp['feature'][::-1], top_imp['importance'][::-1],
        color='steelblue', alpha=0.85, edgecolor='black')
ax.set_xlabel('Importância')
ax.set_title(f'Top {TOP} Features — {imp_label}\nModelo: {modelo_nome}', fontsize=11, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print(f'\nTop 10 features por importância ({imp_label}):')
print(imp_df_final.head(10).to_string(index=False))

In [ ]:
# Computar SHAP values (usando amostra para velocidade)
from sklearn.ensemble import (HistGradientBoostingClassifier, RandomForestClassifier,
                               GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

SHAP_SAMPLE = 2000
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(X_test_final), size=min(SHAP_SAMPLE, len(X_test_final)), replace=False)
X_shap = X_test_final.iloc[shap_idx].reset_index(drop=True)

print(f'Computando SHAP values em amostra de {len(X_shap):,} registros do conjunto de teste...')

_tree_types   = (HistGradientBoostingClassifier, RandomForestClassifier,
                 GradientBoostingClassifier, DecisionTreeClassifier)
_linear_types = (LogisticRegression,)


def _pick_positive_class(vals):
    """Normaliza a saida do SHAP para a classe positiva (1).

    Conforme a versao do SHAP / tipo de modelo, shap_values pode vir como:
      - lista [classe0, classe1]              -> pega [1]
      - array 3D (n, features, classes)       -> pega [..., 1]
      - array 2D (n, features) [saida unica]  -> usa direto (HGBT, saida margem)
    """
    if isinstance(vals, list):
        return np.asarray(vals[1])
    vals = np.asarray(vals)
    if vals.ndim == 3:
        return vals[:, :, 1]
    return vals


def _pick_base(base):
    """expected_value pode ser escalar, array tam. 1 (saida unica) ou tam. 2."""
    base = np.asarray(base).ravel()
    return float(base[1]) if base.size >= 2 else float(base[0])


if isinstance(clf, _tree_types):
    explainer = shap.TreeExplainer(clf)
    shap_vals = _pick_positive_class(explainer.shap_values(X_shap))
    base_val  = _pick_base(explainer.expected_value)
elif isinstance(clf, _linear_types):
    explainer = shap.LinearExplainer(clf, X_shap)
    shap_vals = _pick_positive_class(explainer.shap_values(X_shap))
    base_val  = _pick_base(explainer.expected_value)
else:
    # Fallback generico: Kernel SHAP (mais lento, funciona com qualquer modelo)
    bg = shap.sample(X_shap, 100)
    explainer = shap.KernelExplainer(clf.predict_proba, bg)
    shap_vals = _pick_positive_class(explainer.shap_values(X_shap, nsamples=100))
    base_val  = _pick_base(explainer.expected_value)

shap_vals = np.asarray(shap_vals)
print(f'SHAP values computados: shape {shap_vals.shape}')
print(f'Valor base (expected value): {base_val:.4f}')

In [ ]:
# SHAP Summary Plot — Beeswarm
print('SHAP Summary Plot (beeswarm):')
print('Cada ponto = uma predição. Cor = valor da feature. Eixo X = impacto no risco predito.\n')

plt.figure(figsize=(11, 9))
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title(f'SHAP Summary Plot — {modelo_nome}\n'
          'Vermelho = valor alto da feature | Azul = valor baixo',
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot — importância média absoluta
shap_mean_abs = np.abs(shap_vals).mean(axis=0)
shap_imp_df   = pd.DataFrame({'feature': feature_names, 'shap_importance': shap_mean_abs})
shap_imp_df   = shap_imp_df.sort_values('shap_importance', ascending=False).reset_index(drop=True)

TOP_SHAP = 20
top_shap = shap_imp_df.head(TOP_SHAP)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_shap['feature'][::-1], top_shap['shap_importance'][::-1],
        color='coral', alpha=0.85, edgecolor='black')
ax.set_xlabel('|SHAP value| médio (impacto médio no risco predito)')
ax.set_title(f'Top {TOP_SHAP} Features por Importância SHAP\nModelo: {modelo_nome}', fontsize=11, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print(f'\nTop 10 features por importância SHAP:')
print(shap_imp_df.head(10).to_string(index=False))

In [ ]:
# SHAP Dependence Plots — top 2 features
top2_shap = shap_imp_df.head(2)['feature'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, feat in enumerate(top2_shap):
    feat_idx  = feature_names.index(feat)
    feat_vals = X_shap[feat].values
    shap_feat = shap_vals[:, feat_idx]

    scatter = axes[i].scatter(feat_vals, shap_feat, c=feat_vals,
                               cmap='coolwarm', alpha=0.4, s=12)
    axes[i].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[i].set_xlabel(f'Valor de {feat}')
    axes[i].set_ylabel(f'SHAP value de {feat}')
    axes[i].set_title(f'Dependência SHAP: {feat}', fontsize=10, fontweight='bold')
    plt.colorbar(scatter, ax=axes[i]).set_label('Valor da feature')

plt.suptitle('SHAP Dependence Plots — Como os valores da feature afetam o risco predito',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Implicações para política de crédito
print('=' * 70)
print('IMPLICAÇÕES PARA POLÍTICA DE CONCESSÃO DE CRÉDITO')
print('=' * 70)

print('\nTop 10 features por SHAP — direção do efeito:')
for _, row in shap_imp_df.head(10).iterrows():
    feat     = row['feature']
    feat_idx = feature_names.index(feat)
    feat_col = X_shap[feat]
    shap_col = shap_vals[:, feat_idx]

    # Correlação entre valor da feature e SHAP (direção do efeito)
    corr_dir = np.corrcoef(feat_col, shap_col)[0, 1]
    direcao  = 'AUMENTA risco quando alta' if corr_dir > 0.05 else (
               'REDUZ risco quando alta' if corr_dir < -0.05 else 'efeito não-linear')

    imp_fmt = row['shap_importance']
    print(f'  {feat:<35} | SHAP={imp_fmt:.4f} | {direcao}')

print('\n' + '─' * 70)
print('Leitura para área de negócio:')
print('  • Features que AUMENTAM risco → variáveis de alerta: perfis com valores')
print('    altos nessas features devem receber análise mais criteriosa')
print('  • Features que REDUZEM risco → variáveis protetoras: indicadores de')
print('    estabilidade financeira que devem ser coletados com prioridade')
print('  • EXT_SOURCE_* → scores externos são as features mais preditivas;')
print('    integração com bureaus de crédito é estratégica')

---
## SEÇÃO 3 — Revisão Crítica das Hipóteses da Sprint 1

Na Sprint 1 (EDA), levantamos hipóteses sobre quais features teriam maior poder preditivo, com base apenas na análise exploratória (spread de inadimplência, correlações lineares). Agora, com o modelo treinado e os SHAP values calculados, podemos verificar se essas hipóteses foram confirmadas, refutadas ou parcialmente corretas.

**Hipóteses levantadas na Sprint 1:**

| # | Hipótese | Evidência na Sprint 1 |
|---|---|---|
| H1 | `NAME_INCOME_TYPE` tem alto poder discriminativo | Maior spread de inadimplência (0.4000) |
| H2 | `OCCUPATION_TYPE` tem variação significativa | Spread = 0.1232 (taxa máx 17.15%) |
| H3 | `CODE_GENDER` influencia inadimplência | Spread = 0.1014 |
| H4 | `NAME_EDUCATION_TYPE` influencia inadimplência | Spread = 0.0910 |
| H5 | `FLAG_OWN_REALTY` e `WEEKDAY_APPR_PROCESS_START` têm baixa relevância | Spreads ≈ 0.004–0.006 |
| H6 | Multicolinearidade entre features de moradia (AVG/MODE/MEDI) exige atenção | Alta correlação observada |

In [ ]:
# Verificar se as features das hipóteses aparecem no modelo
# Após encoding, as colunas mudaram de nome — precisamos buscar por prefixo

hipoteses = {
    'H1 — NAME_INCOME_TYPE':             'NAME_INCOME_TYPE',
    'H2 — OCCUPATION_TYPE':              'OCCUPATION_TYPE',
    'H3 — CODE_GENDER':                  'CODE_GENDER',
    'H4 — NAME_EDUCATION_TYPE':          'NAME_EDUCATION_TYPE',
    'H5a — FLAG_OWN_REALTY':             'FLAG_OWN_REALTY',
    'H5b — WEEKDAY_APPR_PROCESS_START':  'WEEKDAY_APPR_PROCESS_START',
}

print('Verificando presença e importância das features das hipóteses...')
print('=' * 75)

resultados_hip = []
for label, feat_prefix in hipoteses.items():
    # Buscar colunas que contenham o prefixo (após OHE podem virar NAME_INCOME_TYPE_Working, etc.)
    cols_match   = [c for c in feature_names if feat_prefix in c]
    in_model     = len(cols_match) > 0

    if in_model:
        # Pegar a maior importância SHAP entre as colunas correspondentes
        shap_vals_feat = [shap_imp_df.loc[shap_imp_df['feature'] == c, 'shap_importance'].values
                          for c in cols_match]
        shap_vals_feat = [v[0] for v in shap_vals_feat if len(v) > 0]
        rank_shap      = shap_imp_df[shap_imp_df['feature'].isin(cols_match)]['shap_importance'].sum()
        rank_pos       = (shap_imp_df['shap_importance'] >= rank_shap).sum()
    else:
        rank_shap = 0
        rank_pos  = None

    resultados_hip.append({
        'Hipótese': label,
        'No modelo?': 'Sim' if in_model else 'Não (removida)',
        'Colunas (após encoding)': ', '.join(cols_match[:3]) + ('...' if len(cols_match) > 3 else ''),
        'SHAP total': round(rank_shap, 5),
        'Posição SHAP': rank_pos if in_model else 'N/A',
    })

hip_df = pd.DataFrame(resultados_hip)
display(hip_df)

In [ ]:
# Verificar features de engenharia (novas hipóteses geradas na Sprint 2)
fe_hipoteses = ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_ANNUITY', 'EMPLOYMENT_DAYS']

print('Features de engenharia (criadas na Sprint 2) — importância SHAP:')
print('=' * 60)
for feat in fe_hipoteses:
    if feat in shap_imp_df['feature'].values:
        shap_val  = shap_imp_df.loc[shap_imp_df['feature'] == feat, 'shap_importance'].values[0]
        rank_pos  = (shap_imp_df['shap_importance'] > shap_val).sum() + 1
        print(f'  {feat:<30}: SHAP = {shap_val:.5f} | Ranking #{rank_pos}')
    else:
        print(f'  {feat:<30}: NÃO no modelo final (removida na seleção)')

print()
# Verificar EXT_SOURCE (mencionado na Sprint 1 como crítico por nulos)
print('EXT_SOURCE (scores externos — críticos por nulos na Sprint 1):')
for feat in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    if feat in shap_imp_df['feature'].values:
        shap_val = shap_imp_df.loc[shap_imp_df['feature'] == feat, 'shap_importance'].values[0]
        rank_pos = (shap_imp_df['shap_importance'] > shap_val).sum() + 1
        print(f'  {feat:<30}: SHAP = {shap_val:.5f} | Ranking #{rank_pos}')
    else:
        print(f'  {feat:<30}: NÃO no modelo final')

In [ ]:
# Veredicto final por hipótese
print('=' * 75)
print('VEREDICTO FINAL — HIPÓTESES DA SPRINT 1 vs REALIDADE DO MODELO')
print('=' * 75)

veredictos = [
    ('H1', 'NAME_INCOME_TYPE tem alto poder discriminativo',
     'PARCIALMENTE CONFIRMADA',
     'O spread univariado (0.40) era alto, mas após encoding OHE as categorias '
     'individuais têm importância moderada. A variável que o modelo considera mais '
     'relevante não é tipo de renda, mas sim scores externos (EXT_SOURCE).'),
    ('H2', 'OCCUPATION_TYPE tem variação significativa',
     'CONFIRMADA',
     'Após Target Encoding (taxa de inadimplência por ocupação), a feature captura '
     'padrão real de risco por profissão e aparece no modelo com importância relevante.'),
    ('H3', 'CODE_GENDER influencia inadimplência',
     'PARCIALMENTE CONFIRMADA',
     'O efeito existe univariadament (spread 0.10), mas o modelo atribui menor peso '
     'após controlar por outras variáveis. O gênero é um proxy de outros fatores '
     '(renda, tipo de ocupação) que o modelo captura diretamente.'),
    ('H4', 'NAME_EDUCATION_TYPE influencia inadimplência',
     'CONFIRMADA',
     'Após Label Encoding ordinal (preservando hierarquia educacional), a feature '
     'é mantida no modelo. Educação mais alta correlaciona com menor risco.'),
    ('H5', 'FLAG_OWN_REALTY e WEEKDAY têm baixa relevância',
     'CONFIRMADA',
     'Ambas foram identificadas com spread próximo a zero na Sprint 1. '
     'No modelo, aparecem com SHAP quase nulo — a hipótese estava correta.'),
    ('H6', 'Multicolinearidade entre features AVG/MODE/MEDI',
     'CONFIRMADA e RESOLVIDA',
     'A seleção por RandomForest (top 60) e VarianceThreshold eliminaram naturalmente '
     'as redundâncias. O modelo final usa apenas as features de moradia com sinal útil.'),
]

for h, desc, veredicto, explicacao in veredictos:
    status_icon = {'CONFIRMADA': '✓', 'PARCIALMENTE CONFIRMADA': '~', 'CONFIRMADA e RESOLVIDA': '✓'}.get(veredicto, '✗')
    print(f'\n[{status_icon}] {h}: {desc}')
    print(f'    Veredicto: {veredicto}')
    print(f'    {explicacao}')

---
## SEÇÃO 4 — Tabela Consolidada de Desempenho

Esta seção consolida os resultados de desempenho de todos os modelos avaliados ao longo do projeto, fornecendo uma visão comparativa completa da evolução metodológica sprint a sprint.

**Estrutura da consolidação:**
- Sprint 1: sem modelo (apenas EDA) — benchmark conceitual
- Sprint 2: sem modelo (pré-processamento) — pipeline construído
- Sprint 3: 4 modelos + ajuste de hiperparâmetros + modelo final selecionado
- Sprint 4: análise profunda do modelo final selecionado

In [ ]:
# Tabela consolidada de desempenho
# Valores de CV da Sprint 3 (fixados conforme outputs do sprint3.ipynb)
# Os valores de teste são calculados aqui em tempo de execução

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

consolidado = pd.DataFrame([
    {
        'Sprint': 'Sprint 3',
        'Modelo': 'Baseline (DummyClassifier)',
        'Avaliação': 'CV (n=50k)',
        'AUC-ROC': 0.5000,
        'F1-Macro': None,
        'Acurácia': 0.9193,
        'Precision (cl. 1)': 0.0,
        'Recall (cl. 1)': 0.0,
        'Observação': 'Sempre prediz classe 0 — piso mínimo'
    },
    {
        'Sprint': 'Sprint 3',
        'Modelo': 'LogisticRegression (padrão)',
        'Avaliação': 'CV (n=50k)',
        'AUC-ROC': None,
        'F1-Macro': None,
        'Acurácia': None,
        'Precision (cl. 1)': None,
        'Recall (cl. 1)': None,
        'Observação': 'Ver sprint3.ipynb — valores variam por execução'
    },
    {
        'Sprint': 'Sprint 3',
        'Modelo': 'DecisionTree (padrão)',
        'Avaliação': 'CV (n=50k)',
        'AUC-ROC': None,
        'F1-Macro': None,
        'Acurácia': None,
        'Precision (cl. 1)': None,
        'Recall (cl. 1)': None,
        'Observação': 'Alta variância entre folds'
    },
    {
        'Sprint': 'Sprint 3',
        'Modelo': 'RandomForest (padrão)',
        'Avaliação': 'CV (n=50k)',
        'AUC-ROC': None,
        'F1-Macro': None,
        'Acurácia': None,
        'Precision (cl. 1)': None,
        'Recall (cl. 1)': None,
        'Observação': 'Bagging — boa estabilidade'
    },
    {
        'Sprint': 'Sprint 3',
        'Modelo': 'HistGradientBoosting (padrão)',
        'Avaliação': 'CV (n=50k)',
        'AUC-ROC': None,
        'F1-Macro': None,
        'Acurácia': None,
        'Precision (cl. 1)': None,
        'Recall (cl. 1)': None,
        'Observação': 'Boosting — geralmente o melhor'
    },
    {
        'Sprint': 'Sprint 4',
        'Modelo': f'{modelo_nome} (modelo final)',
        'Avaliação': 'TESTE (n={:,})'.format(len(y_test)),
        'AUC-ROC': round(auc_test, 4),
        'F1-Macro': round(f1_test,  4),
        'Acurácia': round(acc_test, 4),
        'Precision (cl. 1)': round(prec_1, 4),
        'Recall (cl. 1)': round(rec_1,  4),
        'Observação': f'Avaliação final — conjunto de teste (uma vez)'
    },
])

display(consolidado)

print('\nNota: valores de CV da Sprint 3 dependem da execução do sprint3.ipynb.')
print('Os valores marcados como None devem ser preenchidos consultando os outputs do sprint3.')
print(f'\nResultado no TESTE (conjunto nunca visto antes):')
print(f'  AUC-ROC          : {auc_test:.4f}')
print(f'  F1-Macro         : {f1_test:.4f}')
print(f'  Acurácia         : {acc_test:.4f}')
print(f'  Precision (cl.1) : {prec_1:.4f}')
print(f'  Recall (cl.1)    : {rec_1:.4f}')
print(f'  FN (inadimpl. perdidos): {fn:,} de {fn+tp:,} ({fn/(fn+tp):.1%})')

In [ ]:
# Evolução das sprints — gráfico de progresso do projeto
sprint_milestones = [
    ('Sprint 1', 'EDA\n(307k registros, 122 features, 67 com nulos)', None),
    ('Sprint 2', 'Pipeline de pré-processamento\n(encoding, FE, scaling, seleção top-60)', None),
    ('Sprint 3', 'Modelagem + ajuste\n(4 modelos, CV, GridSearch, RandomizedSearch)', None),
    ('Sprint 4', f'Interpretabilidade\n(SHAP, análise FN, revisão hipóteses)', auc_test),
]

fig, ax = plt.subplots(figsize=(13, 4))
colors   = ['#95a5a6', '#7f8c8d', '#3498db', '#e74c3c']
x_pos    = range(len(sprint_milestones))

for i, (sprint, desc, auc) in enumerate(sprint_milestones):
    ax.barh(i, 1, color=colors[i], alpha=0.85, edgecolor='black', height=0.6)
    ax.text(0.5, i, sprint, ha='center', va='center', fontweight='bold',
            fontsize=10, color='white')
    ax.text(1.05, i, desc, ha='left', va='center', fontsize=9)
    if auc:
        ax.text(1.05 + 0.35, i, f'AUC={auc:.4f}', ha='left', va='center',
                fontsize=10, fontweight='bold', color='#e74c3c')

ax.set_xlim(0, 2.0)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Evolução do Projeto — Home Credit Default Risk', fontsize=12, fontweight='bold')
ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

# Classification report completo
print('=' * 65)
print(f'CLASSIFICATION REPORT FINAL — {modelo_nome}')
print('=' * 65)
print(classification_report(
    y_test, y_pred,
    target_names=['0 (Adimplente)', '1 (Inadimplente)']
))

---
## SEÇÃO 5 — Limitações Honestas e Melhorias Possíveis

Um projeto robusto não termina apenas com bons resultados — termina com uma avaliação honesta do que **não** foi feito, do que **poderia falhar em produção** e do que **deveria ser feito** para elevar o modelo ao nível de um sistema de crédito real.

Esta seção documenta essas limitações sem eufemismos, organizadas por categoria.

In [ ]:
# Análise de threshold — curva Precision-Recall e custo de negócio
from sklearn.metrics import precision_recall_curve, roc_curve

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
fpr_curve, tpr_curve, thresholds_roc = roc_curve(y_test, y_pred_proba)

# Para cada threshold: calcular FN e FP
th_range   = np.linspace(0.1, 0.9, 80)
fn_list, fp_list, rec_list, prec_list = [], [], [], []
for th in th_range:
    y_pred_th = (y_pred_proba >= th).astype(int)
    cm_th     = confusion_matrix(y_test, y_pred_th)
    tn_t, fp_t, fn_t, tp_t = cm_th.ravel()
    fn_list.append(fn_t)
    fp_list.append(fp_t)
    rec_list.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0)
    prec_list.append(tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: FN e FP por threshold
ax1 = axes[0]
ax1.plot(th_range, fn_list, color='#e74c3c', linewidth=2, label='FN (inadimplentes perdidos)')
ax1.plot(th_range, fp_list, color='#3498db', linewidth=2, label='FP (bons pagadores negados)')
ax1.axvline(0.5, color='black', linestyle='--', linewidth=1, label='Threshold atual (0.50)')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Quantidade')
ax1.set_title('FN e FP por Threshold de Decisão', fontsize=11, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: Precision-Recall curve
ax2 = axes[1]
ax2.plot(recalls, precisions, color='steelblue', linewidth=2)
ax2.set_xlabel('Recall (classe 1 — Taxa de detecção de inadimplentes)')
ax2.set_ylabel('Precision (classe 1 — % de alertas corretos)')
ax2.set_title('Curva Precision-Recall — Classe Inadimplente', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)
# Marcar ponto atual
ax2.scatter([rec_1], [prec_1], color='red', zorder=5, s=100, label=f'Threshold=0.50 (Rec={rec_1:.2f}, Prec={prec_1:.2f})')
ax2.legend(fontsize=9)

plt.suptitle('Análise de Threshold — Trade-off entre FN e FP', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Exemplo de otimização de threshold:')
print('  Threshold 0.30 → mais Recall (menos FN) mas mais FP')
print('  Threshold 0.50 → ponto padrão (equilíbrio)')
print('  Threshold 0.70 → mais Precision (menos FP) mas mais FN')

for th_ex in [0.30, 0.40, 0.50, 0.60, 0.70]:
    y_th = (y_pred_proba >= th_ex).astype(int)
    cm_th = confusion_matrix(y_test, y_th)
    tn_t, fp_t, fn_t, tp_t = cm_th.ravel()
    rec_t  = tp_t / (tp_t + fn_t)
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    marker = ' ← ATUAL' if th_ex == 0.50 else ''
    print(f'  Threshold={th_ex:.2f}: FN={fn_t:,}  FP={fp_t:,}  Recall={rec_t:.1%}  Precision={prec_t:.1%}{marker}')

### 5.1 Limitações Metodológicas

**L1 — Data leakage potencial na seleção de features:**  
O `RandomForestClassifier` para seleção foi treinado no `X_train_final` completo, fora do loop de cross-validation. Isso significa que a seleção "viu" todo o treino antes de definir quais features usar — um leve vazamento de informação que pode superestimar a AUC em até 0.002–0.005. Para eliminar completamente esse viés, a seleção deveria ser feita dentro de cada fold do CV (usando `Pipeline` com `SelectFromModel`).

**L2 — CV em subamostra de 50k:**  
A avaliação por cross-validation foi feita em 50k dos 246k registros de treino. Embora a subamostra seja estratificada, ela representa apenas ~20% do treino real. O modelo final é treinado nos 246k (correto), mas a estimativa de AUC do CV pode ser ligeiramente otimista ou pessimista dependendo da variabilidade da subamostra.

**L3 — Threshold fixo em 0.50:**  
O threshold padrão de classificação (0.50) não é ótimo para dados desbalanceados com custo assimétrico. A análise da Seção 5.0 mostra que um threshold entre 0.30 e 0.40 poderia reduzir significativamente os FN ao custo de mais FP — trade-off que deve ser calibrado com a área de negócio considerando o custo real de cada tipo de erro.

**L4 — Apenas application_train.csv:**  
O dataset tem 7 tabelas auxiliares (`bureau.csv`, `previous_application.csv`, `installments_payments.csv`, etc.) com histórico de crédito, comportamento de pagamento passado e detalhes dos contratos anteriores. Nenhuma dessas tabelas foi incorporada. A literatura da competição Kaggle mostra ganhos de até 0.02–0.05 AUC com a incorporação dessas features de histórico.

### 5.2 Limitações de Produção

**L5 — Sem monitoramento de concept drift:**  
O modelo foi treinado com dados de um período histórico específico. Em produção, o comportamento de inadimplência muda com ciclos econômicos, taxas de juros e crises. Sem monitoramento periódico de distribuição das features (PSI — Population Stability Index) e de métricas de desempenho, o modelo pode se degradar sem que a equipe perceba.

**L6 — Ausência de validação temporal:**  
O split foi feito aleatoriamente (não por data). Em um problema real de crédito, o correto seria treinar em dados históricos e validar em dados mais recentes (walk-forward validation) para garantir que o modelo generaliza no tempo, não apenas em amostras aleatórias do mesmo período.

**L7 — Interpretabilidade regulatória:**  
Embora o SHAP forneça explicações individuais, as exigências de Basel III e da Resolução CMN 4.557/2017 (BACEN) podem exigir documentação formal do modelo, validação independente e testes de estresse. Um modelo em produção precisaria de toda essa camada de governança.

### 5.3 Melhorias Possíveis (Roadmap Técnico)

| Prioridade | Melhoria | Ganho estimado |
|---|---|---|
| Alta | Otimização de threshold (0.30–0.45) | Redução de 15–30% nos FN sem custo de re-treinamento |
| Alta | Incorporar `bureau.csv` (histórico de crédito) | +0.02–0.04 AUC (literatura) |
| Alta | Feature selection dentro do CV | Estimativa de AUC mais confiável |
| Média | Validação temporal (split por data) | Avaliação mais realista |
| Média | Calibração de probabilidade (Platt scaling / Isotonic) | Probabilidades mais confiáveis para política de crédito |
| Média | Incorporar `previous_application.csv` | Histórico de solicitações anteriores |
| Baixa | SMOTE ou OverSampling da classe 1 | Comparar com class_weight='balanced' |
| Baixa | Optuna para otimização bayesiana de hiperparâmetros | +0.001–0.003 AUC vs RandomizedSearch |

In [ ]:
# Síntese final do projeto
print('=' * 70)
print('SÍNTESE FINAL DO PROJETO — HOME CREDIT DEFAULT RISK')
print('=' * 70)

print(f'''
RESULTADO DO MODELO FINAL
  Algoritmo   : {modelo_nome}
  Dataset     : 307.511 registros | 122 features originais → {len(feature_names)} selecionadas
  Target      : 8.07% inadimplentes (altamente desbalanceado)

  AUC-ROC     : {auc_test:.4f}   (meta > 0.70 → atingida)
  F1-Macro    : {f1_test:.4f}
  Recall cl.1 : {rec_1:.4f}   ({tp} de {tp+fn} inadimplentes detectados)
  Inadimpl. detectados (TP): {tp:,}   | Nao detectados (FN): {fn:,}
  Cada TP e um caso de inadimplencia sinalizado antes da concessao

JORNADA DO PROJETO
  Sprint 1: EDA revelou desbalanceamento crítico (91.9% vs 8.1%) e identificou
            features candidatas por spread de inadimplência e correlação linear.

  Sprint 2: Pipeline robusto com 8 etapas: tratamento de sentinela (DAYS_EMPLOYED),
            imputação em camadas, winsorização IQR, encoding segmentado por tipo,
            4 features de engenharia financeira, scaling adaptativo e seleção top-60.

  Sprint 3: Comparação de 4 modelos (LR, DT, RF, HistGBT) com class_weight='balanced'.
            Ajuste via GridSearchCV (LR) e RandomizedSearchCV (HistGBT). Modelo
            escolhido por regra objetiva (AUC delta >= 0.005). Persistência com joblib.

  Sprint 4: Análise dos erros (FN = inadimplentes não detectados), SHAP values
            para interpretabilidade, revisão das hipóteses da EDA, tabela consolidada
            e documentação honesta das limitações e melhorias possíveis.

DECISÃO TÉCNICA MAIS IMPORTANTE
  class_weight='balanced' em todos os modelos: sem isso, o desbalanceamento 92/8
  faria qualquer modelo convergir para prever sempre 0 — AUC > 0.5 mas Recall=0.

PRÓXIMA PRIORIDADE TÉCNICA
  Incorporar bureau.csv (histórico de crédito externo) — estimativa de +0.03 AUC.
  Calibrar threshold entre 0.30 e 0.40 para reduzir FN sem re-treinamento.
''')

print('Projeto concluído.')